In [1]:
import pandas as pd
import numpy as np


In [33]:
path_to_trading_folder = 'C:/Users/WilliamFetzner/Documents/Trading/'
backtest = pd.read_excel(f'{path_to_trading_folder}ReportTester-5032285469_3_3_5.xlsx', sheet_name='Deals')
# backtest_fil = backtest[backtest['Direction'] == 'in'].copy()
backtest['Time'] = pd.to_datetime(backtest['Time'], format='%Y.%m.%d %H:%M:%S')
backtest.head()

,Time,Deal,Symbol,Type,Direction,Volume,Price,Order,Commission,Swap,Profit,Balance,Comment
0,2020-01-07 20:00:00,2,EURUSD,sell,in,1,1.11424,2,0,0.0,0,100000.0,NaN
1,2020-01-08 10:13:30,3,EURUSD,buy,out,1,1.11384,3,0,-1.0,40,100039.0,tp 1.11384
2,2020-01-17 16:00:00,4,EURUSD,sell,in,1,1.11004,4,0,0.0,0,100039.0,NaN
3,2020-01-17 16:05:27,5,EURUSD,buy,out,1,1.10969,5,0,0.0,35,100074.0,tp 1.10969
4,2020-01-17 18:00:00,6,EURUSD,sell,in,1,1.10937,6,0,0.0,0,100074.0,NaN


In [21]:
news_events_gr = pd.read_csv(
    f"{path_to_trading_folder}data_files/calendar_df_full_updated.csv", 
    parse_dates=['datetime']
).groupby('datetime').count()
news_events_gr.loc[:, 'event'] = (news_events_gr['Id'] >= 1).astype(int)
news_events_gr_id_drp = news_events_gr.drop(columns='Id')
news_events_gr_id_drp = news_events_gr_id_drp.reset_index()
news_events_gr_id_drp['datetime'] = pd.to_datetime(news_events_gr_id_drp['datetime'], format='%Y-%m-%d %H:%M:%S')

In [23]:
def find_seconds_to_next_news(df, news_counts):
    # Ensure both DataFrames have their time columns as datetime
    df['Time'] = pd.to_datetime(df['Time'])
    news_counts['datetime'] = pd.to_datetime(news_counts['datetime'])

    # Combine both DataFrames
    combined = pd.concat([
        df[['Time']].assign(source='main').rename(columns={'Time': 'time'}),
        news_counts[['datetime']].assign(source='news').rename(columns={'datetime': 'time'})
    ])

    # Sort by time and source
    combined = combined.sort_values(by=['time', 'source'], ascending=[True, False])

    # Find the next and previous news events
    result = combined.copy()
    result['prev_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    result['next_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    
    result['prev_news_time'] = result['prev_news_time'].ffill()
    result['next_news_time'] = result['next_news_time'].bfill()
    result['prev_news_time'] = pd.to_datetime(result['prev_news_time'])
    result['next_news_time'] = pd.to_datetime(result['next_news_time'])

    # Filter for main events and calculate time differences
    result = result[result['source'] == 'main'].copy()
    
    result['time_week_nbr'] = result['time'].dt.isocalendar().week
    result['prev_time_week_nbr'] = result['prev_news_time'].dt.isocalendar().week
    result['next_time_week_nbr'] = result['next_news_time'].dt.isocalendar().week

    # Calculate seconds since last news event
    result['seconds_since_last_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['prev_news_time'].dt.isocalendar().week,
        (result['prev_news_time'] - result['time']).dt.total_seconds() + (2 * 86400),
        (result['prev_news_time'] - result['time']).dt.total_seconds()
    )

    # Calculate seconds to next news event
    result['seconds_to_next_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['next_news_time'].dt.isocalendar().week,
        (result['next_news_time'] - result['time']).dt.total_seconds() - (2 * 86400),
        (result['next_news_time'] - result['time']).dt.total_seconds()
    )

    # Join back to original DataFrame
    final_df = df.merge(
        result[['time', 'seconds_since_last_news_event', 'seconds_to_next_news_event']],
        left_on='Time',
        right_on='time',
        how='left'
    ).drop(columns=['time'])

    return final_df


In [34]:
backtest_w_news = find_seconds_to_next_news(backtest, news_events_gr_id_drp).sort_values('Time')
not_trades = backtest_w_news[(backtest_w_news['Direction'] == 'in') &
                ((backtest_w_news['seconds_since_last_news_event'] > -900) |
                 (backtest_w_news['seconds_to_next_news_event'] < 900))].copy()
not_trades.loc[:, 'close_deal'] = not_trades['Deal'] + 1
profits_to_remove = not_trades['close_deal'].unique()
backtest_w_news[~backtest_w_news['Deal'].isin(profits_to_remove)].Profit.sum()

-2784

In [35]:
backtest_w_news.Profit.sum()

-1848